# Day 27 — The Central Limit Theorem
### Statistics & Probability · Module 2 · Topic 2.7

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · 90 min concepts + 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | **The one distinction** — your data is not your sample mean | 20 min |
| 2 | The theorem, the standard error, and the √n law | 25 min |
| 3 | **How large is large enough?** The n = 30 rule is not true | 20 min |
| 4 | **When it fails** — infinite variance, and dependence | 15 min |
| 5 | What it buys you — a worked confidence interval | 20 min |
| 6 | **Practice notebook (separate file)** | 30 min |

---

## The step we take today

Three sessions have now ended with the same unexplained picture.

- **Day 25:** a Binomial(50, 0.5) looked like a bell curve.
- **Day 25:** a Poisson(λ = 100) looked like a bell curve.
- **Day 26:** twelve uniforms added together looked like a bell curve.

Those three distributions share no mechanism, no shape and no parameters. A binomial counts
successes out of n. A Poisson counts events in an interval. A uniform is a flat line with
nothing interesting about it at all. Each time, the session ended by saying *"Day 27 explains
why"*.

This is Day 27. And the answer is bigger than the curiosity that prompted it: nearly every
confidence interval, p-value and A/B test you will ever run is standing on this one result.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

plt.rcParams["figure.figsize"] = (8, 3.2)
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# 1. ⭐ The one distinction

Before the theorem, the thing people get wrong.

There are **two different distributions** in play and they look nothing alike.

1. The distribution of the **data** — what one observation looks like.
2. The distribution of the **sample mean** — what the *average of n observations* looks like.

The theorem is about the second one. It says nothing whatsoever about the first.

In [ ]:
rng = np.random.default_rng(27)

#  a hard-skewed population: exponential with mean 10
pop = rng.exponential(10, 400_000)

#  and the averages of 30 drawn from it
means30 = rng.exponential(10, (200_000, 30)).mean(axis=1)

print(f"{'':<18} {'mean':>8} {'sd':>8} {'skew':>8}")
print(f"{'ONE value':<18} {pop.mean():>8.3f} {pop.std():>8.3f} {stats.skew(pop):>8.3f}")
print(f"{'AVERAGE of 30':<18} {means30.mean():>8.3f} {means30.std():>8.3f} {stats.skew(means30):>8.3f}")
print()
print(f"theory: sd of the mean = sigma/sqrt(n) = {10/np.sqrt(30):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, d, t, c in [(axes[0], pop, "ONE customer (the raw data)", "#E8674C"),
                    (axes[1], means30, "the AVERAGE of 30 (the sample mean)", "#3E7CB1")]:
    ax.hist(d, bins=90, range=(0, 60), color=c, density=True)
    ax.axvline(10, color="#152238", ls="--", lw=2)
    ax.set_title(t, fontsize=11.5); ax.set_yticks([]); ax.set_xlabel("value")
g = np.linspace(3, 18, 300)
axes[1].plot(g, stats.norm.pdf(g, 10, 10/np.sqrt(30)), color="#152238", lw=2.2)
plt.tight_layout(); plt.show()

print("Same population, same axis. Two completely different distributions.")

## ⚠️ What the theorem does NOT say

Three wrong versions you will hear, and the correct one.

| ❌ Wrong | Why |
|---|---|
| "Big samples are normally distributed" | No. A million salaries are exactly as skewed as a hundred. |
| "With n > 30 I can assume normality" | Of the **mean**, sometimes. Never of the data. |
| "Everything in nature is normal" | Income, city size and file size are all famously not. |

✅ **What it actually says:** take a sample of n, compute its mean, write that one number down.
Do it again. And again. The collection of *those means* is normal — almost regardless of what
the data looked like.

**The data stays exactly as skewed as it always was. Nothing happens to it at all.**

In [ ]:
#  the population's skew is a fixed property. Measure it once, from plenty of data:
big = rng.exponential(10, 2_000_000)
print(f"skew of the POPULATION = {stats.skew(big):.3f}   (exactly 2 in theory)")
print("Collecting more data does not move this. It is not a function of n.")
print()

#  the skew of the MEAN, however, falls away as 1/sqrt(n)
print(f"{'n':>8} {'skew of the mean':>18} {'2/sqrt(n)':>12}")
for n in [1, 5, 30, 100, 1000]:
    m = rng.exponential(10, (60_000, n)).mean(axis=1)
    print(f"{n:>8} {stats.skew(m):>18.3f} {2/np.sqrt(n):>12.3f}")

**How to catch yourself making this mistake:** if you ever run a normality test on your raw
data and then cite the CLT when it fails — stop. The CLT was never a claim about your raw data.
It would not have made that data normal, and it does not need to.

> **A side note you will need in section 3.** A skew *estimate* from a small sample is both
> noisy and biased low — draw 100 points from this population and you will typically measure a
> skew around 1.2, not 2.0. So when we build a rule out of the population skew later, remember
> that the skew itself is something you only know roughly.

---
# 2. The theorem

> Take samples of size n from **any** population with a finite mean μ and a finite standard
> deviation σ. Then the sample means are **normal**, centred on **μ**, with standard deviation
> **σ/√n**.

| Phrase | What it means |
|---|---|
| **any population** | No normality assumed. Skewed, bimodal, discrete — it does not matter. |
| **finite μ and σ** | The one real condition. Section 4 shows what happens without it. |
| **centred on μ** | The sample mean is unbiased — not systematically too high or low. |
| **σ / √n** | The spread. This is the **standard error**, and it is the price list. |

## 2.1 Watch it happen

In [ ]:
rng = np.random.default_rng(271)

fig, axes = plt.subplots(1, 5, figsize=(13, 2.6))
print(f"{'n':>5} {'mean':>8} {'sd':>8} {'theory sd':>10} {'skew':>8} {'theory':>8}")
for ax, n in zip(axes, [1, 2, 5, 30, 100]):
    m = rng.exponential(10, (100_000, n)).mean(axis=1)
    print(f"{n:>5} {m.mean():>8.3f} {m.std():>8.3f} {10/np.sqrt(n):>10.3f} "
          f"{stats.skew(m):>8.3f} {2/np.sqrt(n):>8.3f}")
    ax.hist(m, bins=70, density=True, color="#3E7CB1")
    lo, hi = max(0, 10 - 4.2*10/np.sqrt(n)), 10 + 4.2*10/np.sqrt(n)
    g = np.linspace(lo, hi, 300)
    ax.plot(g, stats.norm.pdf(g, 10, 10/np.sqrt(n)), color="#152238", lw=2)
    ax.set_xlim(0, 40 if n <= 2 else (25 if n <= 5 else 17))
    ax.set_title(f"n = {n}", fontsize=10.5); ax.set_yticks([])
plt.tight_layout(); plt.show()

Two things are happening at once, and the theorem claims both:

- the **shape** becomes normal, and
- the **spread** narrows, as σ/√n.

The theoretical columns are not fitted to the simulation — they are predicted in advance and
the simulation matches them.

## 2.2 "Any population" really does mean any

In [ ]:
rng = np.random.default_rng(272)

pops = {
    "Exponential":     lambda k: rng.exponential(1, k),
    "Bimodal":         lambda k: np.where(rng.random(k) < .5,
                                          rng.normal(-2, .4, k), rng.normal(2, .4, k)),
    "Uniform":         lambda k: rng.uniform(0, 1, k),
    "Lumpy discrete":  lambda k: rng.choice([0, 0, 0, 1, 7], k),
}

fig, axes = plt.subplots(2, 4, figsize=(13, 4.6))
for j, (name, f) in enumerate(pops.items()):
    raw = f(100_000)
    axes[0, j].hist(raw, bins=50, color="#E8674C", density=True)
    axes[0, j].set_title(name, fontsize=11); axes[0, j].set_yticks([])

    m = f(100_000 * 30).reshape(100_000, 30).mean(axis=1)
    axes[1, j].hist(m, bins=60, color="#3E7CB1", density=True)
    g = np.linspace(m.min(), m.max(), 300)
    axes[1, j].plot(g, stats.norm.pdf(g, m.mean(), m.std()), color="#152238", lw=2)
    axes[1, j].set_yticks([]); axes[1, j].set_xlabel("mean of 30", fontsize=9)
axes[0, 0].set_ylabel("population", fontsize=11)
axes[1, 0].set_ylabel("means of 30", fontsize=11)
plt.tight_layout(); plt.show()

Look closely at the **fourth** column. That population takes only three values — 0, 1 and 7 —
and its means at n = 30 are visibly ragged and sit slightly off the curve.

That is worth noticing rather than glossing over. The theorem is an **approximation that
improves with n**, not a switch that flips at some sample size. Section 3 is about exactly how
good the approximation is.

## 2.3 The standard error

In [ ]:
rng = np.random.default_rng(273)

print(f"{'n':>6} {'sd of the means':>18} {'sigma/sqrt(n)':>15}")
for n in [1, 5, 25, 100, 400, 1000]:
    m = rng.exponential(10, (40_000, n)).mean(axis=1)
    print(f"{n:>6} {m.std():>18.4f} {10/np.sqrt(n):>15.4f}")

**Two different numbers, and confusing them is the second most common error in this topic:**

- **σ** is how spread out the **data** is. It does not change when you collect more.
- **SE = σ/√n** is how spread out the **means** are. It shrinks as you collect more.

```python
sd = data.std(ddof=1)
se = sd / np.sqrt(len(data))
```

`ddof=1` because you are estimating σ from a sample — Day 23.

## 2.4 The √n law: precision is expensive

In [ ]:
sigma = 10
print(f"{'n':>7} {'standard error':>16}")
for n in [25, 100, 400, 1600, 6400]:
    print(f"{n:>7} {sigma/np.sqrt(n):>16.3f}")
print()
print("Each row is 4x the data and exactly half the error.")
print("To halve your error bars you need FOUR times the data. Every time.")

This is the arithmetic behind every *"how many users do we need?"* conversation. If someone
asks you to halve the width of a confidence interval, the answer is four times the data — and
that is a budget, not a detail.

---
# 3. ⚠️ How large is large enough?

You have heard "n = 30 is enough". Let us test it properly.

**The test:** take a sample, build a 95% confidence interval, and check whether it actually
contains the true mean. Repeat thousands of times. If the method works, 95% of them should.

In [ ]:
rng = np.random.default_rng(271)

def coverage(sampler, n, mu, trials=4000):
    """How often does a '95%' interval actually contain the truth?"""
    hits = 0
    for _ in range(trials):
        s = sampler(n)
        se = s.std(ddof=1) / np.sqrt(n)
        hits += abs(s.mean() - mu) <= 1.96 * se
    return 100 * hits / trials

cases = [
    ("Normal       skew 0",   lambda n: rng.normal(10, 3, n),    10),
    ("Uniform      skew 0",   lambda n: rng.uniform(0, 20, n),   10),
    ("Exponential  skew 2",   lambda n: rng.exponential(10, n),  10),
    ("Lognormal    skew 6",   lambda n: rng.lognormal(0, 1, n),  np.exp(0.5)),
]

ns = [5, 10, 30, 100, 500]
print(f"{'population':<24}" + "".join(f"{'n='+str(n):>9}" for n in ns))
table = {}
for name, f, mu in cases:
    row = [coverage(f, n, mu) for n in ns]
    table[name] = row
    print(f"{name:<24}" + "".join(f"{v:>8.1f}%" for v in row))
print("\n(every one of these claims to be 95%)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for (name, row), c in zip(table.items(), ["#3E7CB1", "#2FA98C", "#FFC145", "#E8674C"]):
    ax.plot(range(5), row, "o-", color=c, lw=2.2, label=name)
ax.axhline(95, color="#152238", ls="--", lw=1.8)
ax.axvline(2, color="#E8674C", ls=":", lw=2)
ax.text(2.06, 77, 'the "n = 30" rule', color="#E8674C", fontweight="bold", fontsize=10)
ax.set_xticks(range(5)); ax.set_xticklabels([f"n={n}" for n in ns])
ax.set_ylabel("actual coverage of a '95%' interval")
ax.legend(frameon=False, fontsize=9, loc="lower right")
plt.show()

**On lognormal data — which is what income, spend, file size and session length all look like —
your "95%" interval at n = 30 is really an 87% interval.** And even at n = 500 it has only
reached about 94%.

## 3.1 The number that actually decides it

There is an exact rule hiding here:

$$\text{skew of the sample mean} \;=\; \frac{\text{population skew}}{\sqrt{n}}$$

In [ ]:
rng = np.random.default_rng(274)

for name, f, pop_skew in [("exponential", lambda k: rng.exponential(10, k), 2.0),
                          ("lognormal",   lambda k: rng.lognormal(0, 1, k), 6.18)]:
    print(f"\n{name}  (population skew {pop_skew})")
    print(f"  {'n':>6} {'observed':>10} {'skew/sqrt(n)':>14}")
    for n in [5, 30, 100, 400]:
        m = f(60_000 * n).reshape(60_000, n).mean(axis=1)
        print(f"  {n:>6} {stats.skew(m):>10.3f} {pop_skew/np.sqrt(n):>14.3f}")

It is exact, not a rule of thumb. The sample mean inherits the population's skew, divided by √n.

So **"how large is large enough" has a real answer**: large enough that this number is small.
Aim for a mean-skew around 0.3 and the required n falls out:

In [ ]:
print(f"{'population skew':>18} {'n needed':>10}")
for sk in [0.5, 1, 2, 4, 6]:
    print(f"{sk:>18} {(sk/0.3)**2:>10.0f}")
print()
print("A mildly skewed population is fine at a handful of observations.")
print("A heavily skewed one is still not fine at n = 100.")
print("The number 30 appears nowhere in this reasoning.")

---
# 4. ⚠️ When it fails

The theorem needed two things you may not have noticed: a **finite σ**, and **independent**
observations. Both can fail on real data, and neither failure announces itself.

## 4.1 Infinite variance — the mean that never settles

A **Cauchy** distribution looks harmless. It is symmetric and bell-shaped. But its tails are so
heavy that its variance is infinite — not large, *infinite*. So σ/√n is ∞/√n, which is no
promise at all.

In [ ]:
rng = np.random.default_rng(272)

print(f"{'n':>8} {'IQR of the sample mean':>24}")
print("  Cauchy:")
for n in [1, 10, 100, 1000, 10_000]:
    m = stats.cauchy.rvs(size=(10_000, n), random_state=rng).mean(axis=1)
    print(f"  {n:>8} {np.percentile(m,75)-np.percentile(m,25):>24.4f}")
print("  Normal, for comparison:")
for n in [1, 10, 100, 1000]:
    m = rng.normal(0, 1, (10_000, n)).mean(axis=1)
    print(f"  {n:>8} {np.percentile(m,75)-np.percentile(m,25):>24.4f}")

**The Cauchy IQR is the same at n = 1 and at n = 10,000.** Collecting more data buys you
nothing at all. The mean of n Cauchy samples is itself Cauchy, with exactly the same spread.

In [ ]:
rng = np.random.default_rng(99)
N = 4000
fig, axes = plt.subplots(1, 2, figsize=(11, 2.9), sharex=True)
for ax, name, draws, c in [
        (axes[0], "Normal data", rng.normal(0, 1, (6, N)), "#3E7CB1"),
        (axes[1], "Cauchy data (infinite variance)",
         stats.cauchy.rvs(size=(6, N), random_state=rng), "#E8674C")]:
    for row in draws:
        ax.plot(np.cumsum(row)/np.arange(1, N+1), color=c, lw=1, alpha=0.75)
    ax.axhline(0, color="#152238", ls="--", lw=1.6)
    ax.set_title(name, fontsize=11.5); ax.set_xlabel("observations collected")
axes[0].set_ylim(-1.2, 1.2); axes[1].set_ylim(-12, 12)
axes[0].set_ylabel("running average")
plt.tight_layout(); plt.show()

print("Left: the average settles. Right: it never does, at any sample size.")

**What happens in practice:** one enormous observation arrives and drags the average with it.
Then a bigger one arrives. The average never converges because the next outlier is always big
enough to move it.

**Real data that behaves this way:** insurance claims (one hurricane dwarfs a decade), wealth,
city and firm sizes, financial returns in a crisis, file sizes, network traffic bursts.

**The practical test:** plot the running mean of your own data. If it is still jumping after
thousands of points, the mean may not be a sensible summary at all — use the median or a
trimmed mean, and say so.

## 4.2 Dependence — when your n is not your n

In [ ]:
rng = np.random.default_rng(274)
n, rho = 100, 0.8

#  independent
ind = rng.normal(0, 1, (40_000, n)).mean(axis=1)

#  correlated: today partly repeats yesterday (AR(1)), same marginal sd of 1
e = rng.normal(0, np.sqrt(1 - rho**2), (40_000, n))
x = np.zeros_like(e)
x[:, 0] = rng.normal(0, 1, 40_000)
for t in range(1, n):
    x[:, t] = rho * x[:, t-1] + e[:, t]
dep = x.mean(axis=1)

print(f"each series has sd 1:  independent {rng.normal(0,1,100_000).std():.3f}   "
      f"correlated {x.std():.3f}")
print()
print(f"CLT predicts a standard error of sigma/sqrt(n) = {1/np.sqrt(n):.4f}")
print(f"  independent, measured: {ind.std():.4f}   <- correct")
print(f"  correlated,  measured: {dep.std():.4f}   <- {dep.std()/ind.std():.1f}x too wide")
print()
eff = n / (dep.std()*np.sqrt(n))**2
print(f"Effective sample size: about {eff:.0f}, not {n}.")

If today's value partly repeats yesterday's, each new reading carries less new information than
an independent one would. Here σ/√n says the standard error is 0.10; the truth is 0.29.

**Every confidence interval built on it is three times too narrow, and every p-value far too
small.** This is the failure you are most likely to hit in practice, because almost all time
series and repeated-measures data is correlated — and nothing in the analysis warns you.

## 4.3 The checklist

| # | Check | How |
|---|---|---|
| 1 | Is σ finite? | Plot the running mean. Still jumping after thousands of points → stop. |
| 2 | Are the observations independent? | Repeated measures, or anything over time, usually are not. |
| 3 | How skewed is the population? | `stats.skew(data)`, then n ≈ (skew / 0.3)² |
| 4 | Am I talking about the data or the mean? | The theorem only licensed a claim about the mean. |

Checks 1 and 2 are **pass/fail** — if they fail, the theorem gives you nothing. Check 3 is a
dial, not a gate. Check 4 is free and catches the most common error of all.

---
# 5. What it buys you

## The business question

*"What is our average customer worth?"*

Spend per customer is heavily skewed — most customers spend a little, a few spend a lot. It is
lognormal, with a skew around 4.8, and it will never be normal no matter how much you collect.

In [ ]:
rng = np.random.default_rng(2701)
mu_log, sig_log = 3.2, 0.9
population = rng.lognormal(mu_log, sig_log, 300_000)
true_mean = np.exp(mu_log + sig_log**2 / 2)

print(f"true mean   {true_mean:8.2f}")
print(f"median      {np.median(population):8.2f}   <- the TYPICAL customer")
print(f"sd          {population.std():8.2f}")
print(f"skew        {stats.skew(population):8.2f}")
print(f"top 1% above{np.percentile(population, 99):8.2f}")
print()
print("The mean is well above the median - that is what a long right tail does.")

In [ ]:
#  we survey 200 customers
n = 200
survey = np.random.default_rng(99)          # a dedicated seed, so this is quotable
sample = survey.choice(population, n, replace=False)

xbar = sample.mean()
sd   = sample.std(ddof=1)
se   = sd / np.sqrt(n)
lo, hi = xbar - 1.96*se, xbar + 1.96*se

print(f"sample mean  {xbar:8.2f}")
print(f"sample sd    {sd:8.2f}")
print(f"standard err {se:8.3f}   = sd / sqrt(200)")
print()
print(f"95% CI:  {lo:.2f}  to  {hi:.2f}")
print(f"true mean is {true_mean:.2f}  ->  {'inside' if lo <= true_mean <= hi else 'MISSED'}")

In [ ]:
#  the sample itself is wildly non-normal - and it does not matter
print(f"skew of the sample      {stats.skew(sample):.2f}")
print(f"Shapiro-Wilk p-value    {stats.shapiro(sample).pvalue:.2e}   <- fails hard")
print()
print("That test asked whether the DATA is normal. It is not, and never will be.")
print("The interval above does not need it to be.")

In [ ]:
#  what IS normal: the distribution of the mean
means = np.array([rng.choice(population, n).mean() for _ in range(15_000)])

print(f"distribution of the mean of {n}:")
print(f"  mean {means.mean():.3f}   (true {true_mean:.3f})")
print(f"  sd   {means.std():.3f}   (theory sigma/sqrt(n) = {population.std()/np.sqrt(n):.3f})")
print(f"  skew {stats.skew(means):+.3f}   (theory {stats.skew(population)/np.sqrt(n):+.3f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(population, bins=120, range=(0, 200), color="#E8674C", density=True)
axes[0].axvline(np.median(population), color="#2FA98C", lw=2.2)
axes[0].axvline(true_mean, color="#152238", ls="--", lw=2.2)
axes[0].set_title(f"spend per customer (skew {stats.skew(population):.1f})", fontsize=11)
axes[1].hist(means, bins=60, color="#3E7CB1", density=True)
g = np.linspace(means.min(), means.max(), 300)
axes[1].plot(g, stats.norm.pdf(g, means.mean(), means.std()), color="#152238", lw=2.2)
axes[1].set_title(f"the mean of {n} (skew {stats.skew(means):.2f})", fontsize=11)
for ax in axes:
    ax.set_yticks([])
plt.tight_layout(); plt.show()

Note that 4.8 / √200 = 0.34, which is what the right-hand panel came out at. The skew rule from
section 3 predicted it before we drew it.

## ⚠️ What "95% confident" actually means

In [ ]:
rng = np.random.default_rng(2702)
fig, ax = plt.subplots(figsize=(8, 3.4))
shown, misses = 40, 0
for i in range(shown):
    s = rng.choice(population, n)
    se_ = s.std(ddof=1)/np.sqrt(n)
    l, h = s.mean() - 1.96*se_, s.mean() + 1.96*se_
    hit = l <= true_mean <= h
    misses += not hit
    ax.plot([l, h], [i, i], color="#2FA98C" if hit else "#E8674C", lw=2.2)
ax.axvline(true_mean, color="#152238", ls="--", lw=2.2)
ax.set_yticks([]); ax.set_xlabel("average spend")
ax.set_title(f"40 surveys, each with its own 95% interval - {misses} missed")
plt.show()

print(f"{misses} of {shown} missed. At 95%, about 1 in 20 is the deal you signed.")

**"95% confident" means:** if you repeated the whole survey forever, 95% of the intervals you
built would contain the truth.

**It does NOT mean:**

- *"a 95% chance the true mean is in THIS interval"* — your interval either contains it or it
  does not; there is no chance left in it.
- *"95% of customers spend between 33 and 41"* — that is a statement about **people**. This is
  a statement about an **average**.

The second misreading is the one that reaches newspapers.

## Is n = 200 actually enough here?

Section 3 said to check rather than assume. The population skew is 4.8, so:

In [ ]:
sk_true = (np.exp(0.9**2) + 2) * np.sqrt(np.exp(0.9**2) - 1)    # exact for a lognormal
sk_est  = stats.skew(population)
print(f"population skew: exact {sk_true:.2f}, measured {sk_est:.2f}")
print(f"n needed for a mean-skew of 0.3:  {(sk_true/0.3)**2:.0f}")
print(f"we used                           {n}")
print()
#  measure the real coverage
for nn in [30, 100, 200, 1000]:
    hits = 0
    for _ in range(4000):
        s = rng.choice(population, nn)
        se_ = s.std(ddof=1)/np.sqrt(nn)
        hits += abs(s.mean() - true_mean) <= 1.96*se_
    print(f"  n = {nn:>5}:  a '95%' interval covers {100*hits/4000:.1f}% of the time")

Honest answer: **n = 200 gives about 92%, not 95%.** The interval is too narrow, for two
reasons that compound: the population is skewed enough that 200 is short of the ~250 the rule
asks for, *and* a small sample from a heavy-tailed population systematically **underestimates
σ**, which narrows the interval further.

Notice that even n = 1000 only reaches about 93%. That is worth reporting, not hiding.

And one more caveat on the rule itself: look again at how noisy a skew *estimate* is on
heavy-tailed data. Five independent samples of 1,000 from this population gave skew estimates
ranging from 2.7 to 6.5 — so `n ≈ (skew/0.3)²` could have told you anything from 80 to 470.
The rule is a far better guide than "n = 30", and it is still built on a quantity you cannot
pin down from a small sample.

If you need genuine 95% coverage on data like this, the answer is a **bootstrap** rather than a
bigger n — which is what Part D of the practice notebook asks you to build.

## Why this matters

| Technique | What it inherits from today |
|---|---|
| Confidence intervals | Every ± you have ever seen reported |
| t-tests and z-tests | "Is this difference real?" — Day 28 |
| A/B testing | Day 25's checkout experiment, done properly |
| Polling margins of error | ±3% is σ/√n with n ≈ 1,000 |
| Quality control limits | Day 24's ±3σ charts, applied to batch means |
| Regression standard errors | Module 3, every coefficient |

Which is why the failure cases are not academic. When the CLT quietly does not hold, **none of
these methods announce it**. They all return a number, and the number is too confident.

---
# 6. Mini build — a CLT health check

Three numbers instead of one rule.

In [ ]:
def clt_check(data, n):
    """Can I trust a CLT-based interval built from n of these?"""
    d  = np.asarray(data, dtype=float)
    sk = stats.skew(d)

    #  1. does the mean settle at all?  (catches heavy tails)
    #     scale the wobble by the data's own IQR, not by the mean - a mean
    #     near zero, or shifted far from zero, would make |mean| useless here
    run   = np.cumsum(d) / np.arange(1, len(d) + 1)
    tail  = run[len(d)//2:]
    scale = np.percentile(d, 75) - np.percentile(d, 25)
    drift = (tail.max() - tail.min()) / scale

    #  2. how skewed will the sample mean be?
    mean_skew = sk / np.sqrt(n)

    #  3. what n would this population actually need?
    n_needed = (sk / 0.3) ** 2

    print(f"  population skew      {sk:10.2f}")
    print(f"  running-mean drift   {drift:10.3f}   (of one IQR)")
    print(f"  skew of mean(n={n})  {mean_skew:10.2f}")
    print(f"  n for a skew of 0.3  {n_needed:10.0f}")
    ok = abs(mean_skew) < 0.3 and drift < 0.10
    print(f"  verdict: {'OK' if ok else 'use a bootstrap, or a bigger n'}")
    return ok

rng = np.random.default_rng(275)
for name, d in [("normal     ", rng.normal(10, 3, 20_000)),
                ("exponential", rng.exponential(10, 20_000)),
                ("lognormal  ", rng.lognormal(0, 1, 20_000)),
                ("Cauchy     ", stats.cauchy.rvs(size=20_000, random_state=rng) + 50)]:
    print(f"\n{name}  with n = 30")
    clt_check(d, 30)

Six things worth noticing:

1. **It checks rather than assumes** — three numbers beat one memorised rule.
2. **Drift catches heavy tails** — a mean that will not settle.
3. **skew/√n is the real criterion**, not the sample size alone.
4. **`n_needed` is actionable** — it tells you what to go and collect.
5. **It cannot see dependence.** Nothing you compute from one sample can.
6. **Run it before every interval.** It costs one line.

Point 5 matters: only knowing *how the data was collected* will tell you whether the
observations were independent. A green verdict here is not full clearance.

---

# Debug corner — eight CLT mistakes

| You did | What goes wrong | Fix |
|---|---|---|
| said "n > 30, so my data is normal" | the data never becomes normal | the theorem is about the **mean** |
| tested the raw data for normality | it fails, and it was never relevant | ask about the mean instead |
| divided σ by n | the error is far too small | divide by **√n** |
| used σ where you needed SE | σ describes people, SE describes the average | SE = σ/√n |
| trusted n = 30 on skewed data | a "95%" interval covering 87% | n ≈ (skew / 0.3)² |
| averaged heavy-tailed data | the mean never settles | plot the running mean; use a median |
| used a time series as if independent | intervals 3× too narrow | effective n, not n |
| read a CI as "95% of customers" | it is about the mean, not people | say "the average is between…" |

Rows 1 and 2 are the same misunderstanding, and between them they account for most of the
trouble people have with this topic.

In [ ]:
#  proof that two of those really do go wrong
rng = np.random.default_rng(276)
d = rng.exponential(10, 200)

print("dividing by n instead of sqrt(n):")
print(f"  correct  sd/sqrt(n) = {d.std(ddof=1)/np.sqrt(200):.4f}")
print(f"  wrong    sd/n       = {d.std(ddof=1)/200:.4f}   <- 14x too small")
print()
print("confusing sigma with SE:")
print(f"  sigma (spread of PEOPLE)   = {d.std(ddof=1):.2f}")
print(f"  SE    (spread of AVERAGES) = {d.std(ddof=1)/np.sqrt(200):.2f}")

---
# Recap — today in twelve lines

1. ⭐ Your **data** and your **sample mean** are two different distributions.
2. The theorem is about the mean. It says nothing about the data.
3. Sample means are normal, centred on μ, with sd σ/√n — from **any** population.
4. σ/√n is the **standard error**. σ describes people; SE describes the average.
5. √n law: 4× the data buys 2× the precision. That is a budget.
6. ⚠️ The data never becomes normal. A million salaries are still skewed.
7. ⚠️ "n = 30 is enough" is false. On lognormal data it gives an 87% interval.
8. skew of the mean = population skew / √n. Exact, and it decides n.
9. A defensible rule: **n ≈ (skew / 0.3)²**.
10. ⚠️ Fails with infinite variance — a Cauchy mean never settles, at any n.
11. ⚠️ Fails with dependence — 100 correlated points were worth 12.
12. Nearly every interval, p-value and A/B test ahead of you rests on this.

### Homework
- Run `clt_check` on every numeric column of your Day 21 project data and say which ones you
  could safely build an interval from.
- Find a published ± figure — a poll, a survey, a trial — and work out what n it implies.
- Take one skewed column and find the n its skew actually demands.

### Next class — Topic 2.8: Hypothesis Testing
Today you learned how far a sample mean typically lands from the truth. Next class turns that
into a decision procedure: *is this difference real, or is it the √n noise we just measured?*

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*